<center><h1>Kinetic Rate Laws</h1></center>
<hr/>

Assigns a kinetic rate-law expression to every PSS reaction:

| Step | Description |
|---|---|
| **Load** | Read `reactions_by_combination.xlsx` — maps each reaction ID to an SBO term and a functional form string |
| **Parse** | Replace symbolic tokens (`S`, `P`, `M`, `Kd_S`, …) with actual species IDs from the PSS reaction data |
| **Fallback** | Reactions whose functional form cannot be fully resolved default to a simple degradation law `−k_deg·[S]` |
| **Save** | Serialise the `rate_law(reaction_id)` callable with `dill` |

> **TODO**: resolve remaining reactions that fall through to the degradation fallback — see `make_rate_law_mapper` comments.

In [1]:
import re
import json
from pathlib import Path
from typing import Optional

import dill
import numpy as np
import pandas as pd

from scripts.utils import (
    PSS_REACTIONS_PATH,
    PSS_SPECIES_PATH,
    REACTIONS_BY_COMBINATION_PATH,
    REACTIONS_BY_COMBINATION_SHEET,
    KINETIC_RATE_LAWS_PATH,
    RED, RESET,
)

## Load

In [2]:
reactions = json.loads(Path(PSS_REACTIONS_PATH).read_text())
species   = json.loads(Path(PSS_SPECIES_PATH).read_text())

print(f"Loaded {RED}{len(species)}{RESET} species and {RED}{len(reactions)}{RESET} reactions.")

Loaded 1139 species and 864 reactions.


In [3]:
excel = pd.read_excel(
    REACTIONS_BY_COMBINATION_PATH,
    sheet_name=REACTIONS_BY_COMBINATION_SHEET,
    skiprows=0,
)
excel.head()

,reaction_type,reaction_mechanism,reaction_effect,#_substrate,#_product,#_modifer,reaction_count,Comment - SP,decision based on,reaction_process,...,modifier role_SBO,reaction_relationship,reaction_relationship_SBO,rate_law,rate_law_SBO,reversible_(Y/N),fast_slow,Function_form,Source,reaction_ids
0,binding/oligomerisation,inhibition by binding,inhibition,2,1,0,5,NaN,is binding,molecular interaction,...,NaN,inhibition,SBO_0000169,common modular rate law,SBO_0000528,Y,fast,dP=(((k_cat*S1*S2)/(Kd_S1*Kd_S2))-(P/(Kd_P*k_c...,https://doi.org/10.1093/bioinformatics/btq141,"rx00581, rx00459, rx00446, rx00444, rx00207"
1,binding/oligomerisation,inhibition of AGO by binding,inhibition,2,1,0,1,NaN,is binding,molecular interaction,...,NaN,inhibition,SBO_0000169,common modular rate law,SBO_0000528,Y,fast,dP=(((k_cat*S1*S2)/(Kd_S1*Kd_S2))-(P/(Kd_P*k_c...,NaN,rx00199
2,binding/oligomerisation,with catalyst,activation,1,1,1,1,MOVED FROM rx00917,binding with modifier,molecular interaction,...,NaN,stimulation,SBO_0000170,Henri-Michaelis-Menten rate law,SBO_0000029,Y,fast,dP=(k_cat*[M]*[S])/([S] + Kd_S),NaN,rx00094
3,binding/oligomerisation,NaN,activation,1,1,1,2,"ADDED rx00987, MOVED FROM rx00942",binding with modifier,molecular interaction,...,NaN,stimulation,SBO_0000170,Henri-Michaelis-Menten rate law,SBO_0000029,Y,fast,dP=(k_cat*[M]*[S])/([S] + Kd_S),NaN,rx00987
4,binding/oligomerisation,NaN,activation,2,1,0,97,"MOVED TO rx00917, rx00942",is binding,molecular interaction,...,NaN,stimulation,SBO_0000170,common modular rate law,SBO_0000528,Y,fast,dP=(((k_cat*S1*S2)/(Kd_S1*Kd_S2))-(P/(Kd_P*k_c...,https://doi.org/10.1093/bioinformatics/btq141,"rx00985, rx00983, rx00981, rx00980, rx00979, r..."


## Build Reaction → (SBO, Functional Form) Map

In [4]:
# Map each reaction ID to its (SBO term, functional form string).
# Warn on duplicate IDs — last entry wins.
rid_sbo: dict[str, tuple[str, str]] = {}
for sbo, ids, func_form in excel[['rate_law_SBO', 'reaction_ids', 'Function_form']].values:
    for rid in ids.split(', '):
        rid = rid.strip()
        if rid in rid_sbo:
            print(f"⚠️  Duplicate reaction ID: {RED}{rid}{RESET} (SBO={sbo})")
        rid_sbo[rid] = (sbo, func_form)

print(f"Mapped {RED}{len(rid_sbo)}{RESET} reaction IDs to rate laws.")

Mapped 894 reaction IDs to rate laws.


### Example

In [5]:
# Inspect the first reaction
first_rxn_id = next(iter(reactions))
print(f"Reaction {RED}{first_rxn_id}{RESET}: {rid_sbo.get(first_rxn_id, 'not found')}")

Reaction rx00669: ('SBO_0000528', 'dP=(((k_cat*S1*S2)/(Kd_S1*Kd_S2))-(P/(Kd_P*k_cat)))/((1+S1/Kd_S1)*(1+S2/Kd_S2)+(1+P/Kd_P) - 1)')


## Parse Rate Laws

Replace symbolic tokens in functional form strings with actual PSS species IDs:

| Token | Maps to |
|---|---|
| `S`, `S2`, … | reactants[0], reactants[1], … |
| `P`, `P2`, … | products[0], products[1], … |
| `M`, `M2`, … | modifiers[0], modifiers[1], … |
| `Kd_S`, `Kd_M`, … | `Kd_<species_id>` |

Raises `ValueError` if any RHS token cannot be resolved.

In [6]:
def replace_species(equation: str, reaction_id: str, reaction_data: dict) -> str:
    """Replace S/P/M tokens in a kinetic equation RHS with PSS species IDs.

    Only tokens on the RHS are validated. A `dP` on the LHS is ignored —
    the function returns only the RHS expression, ready for use in Antimony.

    Args:
        equation:      Functional form string, e.g. 'dP = kcat*[M]*[S] / ([S] + Kd_S)'.
        reaction_id:   Reaction ID (used in error messages only).
        reaction_data: Dict with keys 'reactants', 'products', 'modifiers'.

    Returns:
        Cleaned RHS string with species IDs substituted.

    Raises:
        ValueError: If the equation has no '=', or if any RHS token is unmapped.
    """
    reactants = reaction_data.get('reactants', []) or []
    products  = reaction_data.get('products',  []) or []
    modifiers = reaction_data.get('modifiers', []) or []

    def lookup(token: str) -> Optional[str]:
        """Resolve a bare S/P/M token to the corresponding species ID."""
        m = re.fullmatch(r'([SPM])(\d*)', token)
        if not m:
            return None
        letter, num = m.groups()
        idx = (int(num) if num else 1) - 1
        if letter == 'S': return reactants[idx] if idx < len(reactants) else None
        if letter == 'P': return products[idx]  if idx < len(products)  else None
        if letter == 'M': return modifiers[idx] if idx < len(modifiers) else None
        return None

    if '=' not in equation:
        raise ValueError(f"Reaction {reaction_id}: equation has no '=' → {equation!r}")

    _, rhs = equation.split('=', 1)

    used:   set[str] = set()
    mapped: set[str] = set()

    def repl(match) -> str:
        token = match.group(0)
        if token.startswith('Kd_'):
            base    = token[3:]
            used.add(base)
            species = lookup(base)
            if species:
                mapped.add(base)
                return f'Kd_{species}'
            return token
        used.add(token)
        species = lookup(token)
        if species:
            mapped.add(token)
            return species
        return token

    new_rhs  = re.sub(r'\bKd_[SPM]\d*\b|\b[SPM]\d*\b', repl, rhs)
    unmapped = sorted(used - mapped)

    if unmapped:
        raise ValueError(
            f"Reaction {reaction_id}: unmapped tokens → {unmapped}"
        )

    return (
        new_rhs
        .replace('k_cat', 'kcat')
        .replace('**',    '^')
        .replace('[',     '')
        .replace(']',     '')
    )

### Example usage

In [7]:
# Show the parsed rate law for the first reaction
first_rxn_id = next(iter(reactions))
sbo, func_form = rid_sbo[first_rxn_id]
parsed = replace_species(func_form, first_rxn_id, reactions[first_rxn_id])
print(f"Reaction : {RED}{first_rxn_id}{RESET}")
print(f"Raw form : {func_form}")
print(f"Parsed   : {parsed}")

Reaction : rx00669
Raw form : dP=(((k_cat*S1*S2)/(Kd_S1*Kd_S2))-(P/(Kd_P*k_cat)))/((1+S1/Kd_S1)*(1+S2/Kd_S2)+(1+P/Kd_P) - 1)
Parsed   : (((kcat*s_VPg_cyt_p*s_LHB1B1_cyt_p)/(Kd_s_VPg_cyt_p*Kd_s_LHB1B1_cyt_p))-(s_LHB1B1VPg_cyt_c/(Kd_s_LHB1B1VPg_cyt_c*kcat)))/((1+s_VPg_cyt_p/Kd_s_VPg_cyt_p)*(1+s_LHB1B1_cyt_p/Kd_s_LHB1B1_cyt_p)+(1+s_LHB1B1VPg_cyt_c/Kd_s_LHB1B1VPg_cyt_c) - 1)


## Build Rate-Law Mapper

In [8]:
def build_rate_law_mapper(rid_sbo: dict, reactions: dict) -> callable:
    """Build a self-contained rate-law callable and return it.

    All data is captured in the closure at build time so the returned
    callable can be serialised with dill and loaded independently.

    Args:
        rid_sbo:   {reaction_id: (sbo, func_form)} mapping.
        reactions: PSS reactions dict keyed by reaction ID.

    Returns:
        rate_law(reaction_id: str) → str | None

    Raises:
        ValueError: Propagated from replace_species if any RHS token is unmapped.
        KeyError:   If reaction_id is not found in rid_sbo or reactions.
    """
    _rid_sbo   = dict(rid_sbo)
    _reactions = {rid: dict(r) for rid, r in reactions.items()}

    def rate_law(reaction_id: str) -> Optional[str]:
        """Return the parsed kinetic rate-law RHS for *reaction_id*.

        Returns None only if the functional form is missing (NaN) in the
        Excel sheet — all other failure modes raise.
        """
        func_form = _rid_sbo[reaction_id][1]
        if isinstance(func_form, float) and np.isnan(func_form):
            return None

        return replace_species(func_form, reaction_id, _reactions[reaction_id])

    return rate_law

In [9]:
rate_law_mapper = build_rate_law_mapper(rid_sbo, reactions)

### Inspect all rate laws

In [10]:
resolved = []
missing  = []
errors   = []

for rxn_id in reactions:
    try:
        eq = rate_law_mapper(rxn_id)
        if eq is None:
            missing.append(rxn_id)
        else:
            resolved.append((rxn_id, eq))
    except (ValueError, KeyError) as e:
        errors.append((rxn_id, str(e)))

print(f"Resolved : {RED}{len(resolved)}{RESET}")
print(f"Missing  : {RED}{len(missing)}{RESET}  (NaN functional form in Excel)")
print(f"Errors   : {RED}{len(errors)}{RESET}  (unmapped tokens / not in rid_sbo)")

if errors:
    print(f"\nFailed reactions:")
    for rxn_id, msg in errors:
        print(f"  {msg}")

Resolved : 864
Missing  : 0  (NaN functional form in Excel)
Errors   : 0  (unmapped tokens / not in rid_sbo)


## Save

In [11]:
with open(KINETIC_RATE_LAWS_PATH, 'wb') as f:
    dill.dump(rate_law_mapper, f)

print(f"Saved rate-law mapper → {KINETIC_RATE_LAWS_PATH}")

Saved rate-law mapper → /home/ratanasoski/Desktop/Master_Thesis/ThesisRepository/results/rate_law_mapper.pkl
